# Datenpipeline Diagnose

Kombiniertes Notebook für die drei Diagnose-Checks der Datenpipeline:
1. **Status-Check**: Fortschritt der Fixtures- und Match-Stats-Abrufe pro Team
2. **Stats-Abdeckung**: welche Statistik-Typen (Ballbesitz, Ecken etc.) sind zuverlässig genug für Features?
3. **Feature-Sanity-Check**: Klassenverteilung und fehlende Werte in der finalen `match_features.csv`

Ersetzt die Einzel-Skripte `check_status.py`, `analyze_stats_coverage.py`, `quick_check_features.py`.

In [ ]:
import sys
import json
import csv
from pathlib import Path
from collections import Counter, defaultdict

# src/ zum Pfad hinzufügen, damit config.py etc. importierbar sind
sys.path.insert(0, str(Path.cwd().parent / "src"))

from config import DATA_RAW_DIR, DATA_PROCESSED_DIR, WM2026_TEAMS

STATS_DIR = DATA_RAW_DIR / "stats"
FEATURES_FILE = DATA_PROCESSED_DIR / "match_features.csv"


def _read_json_robust(path: Path):
    """Liest JSON ein, mit Fallback auf cp1252 (ältere Dateien vor Encoding-Fix)."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except UnicodeDecodeError:
        return json.loads(path.read_text(encoding="cp1252"))

## 1. Status-Check: Fixtures & Match-Stats pro Team

In [ ]:
def load_existing_stats_ids() -> set:
    if not STATS_DIR.exists():
        return set()
    ids = set()
    for stats_file in STATS_DIR.glob("*.json"):
        try:
            ids.add(int(stats_file.stem))
        except ValueError:
            continue
    return ids


def check_status():
    existing_stats_ids = load_existing_stats_ids()

    rows = []
    total_fixtures = 0
    total_unique_ids = set()

    for fifa_name in WM2026_TEAMS:
        fixtures_file = DATA_RAW_DIR / f"fixtures_{fifa_name.replace(' ', '_')}.json"

        if not fixtures_file.exists():
            rows.append((fifa_name, "FEHLT", "-", "-"))
            continue
        if fixtures_file.stat().st_size == 0:
            rows.append((fifa_name, "LEER", "-", "-"))
            continue
        try:
            fixtures = _read_json_robust(fixtures_file)
        except json.JSONDecodeError:
            rows.append((fifa_name, "KAPUTT", "-", "-"))
            continue

        fixture_ids = [fx.get("fixture", {}).get("id") for fx in fixtures]
        fixture_ids = [fid for fid in fixture_ids if fid is not None]

        n_games = len(fixture_ids)
        n_stats_done = sum(1 for fid in fixture_ids if fid in existing_stats_ids)
        pct = round(100 * n_stats_done / n_games, 0) if n_games > 0 else 0

        rows.append((fifa_name, str(n_games), str(n_stats_done), f"{pct:.0f}%"))
        total_fixtures += n_games
        total_unique_ids.update(fixture_ids)

    print(f"{'Team':<26} {'Spiele':>8} {'Stats':>8} {'Fortschr.':>10}")
    print("-" * 56)
    for fifa_name, n_games, n_stats, pct in rows:
        print(f"{fifa_name:<26} {n_games:>8} {n_stats:>8} {pct:>10}")

    print("-" * 56)
    n_done_teams = sum(1 for r in rows if r[1] not in ("FEHLT", "LEER", "KAPUTT"))
    print(f"\nTeams komplett abgerufen: {n_done_teams} / {len(WM2026_TEAMS)}")
    print(f"Total Spiele (mit Duplikaten über Teams): {total_fixtures}")
    print(f"Eindeutige Fixtures: {len(total_unique_ids)}")
    overall_pct = round(100 * len(existing_stats_ids & total_unique_ids) / len(total_unique_ids), 1) if total_unique_ids else 0
    print(f"Match-Stats gesamt: {len(existing_stats_ids & total_unique_ids)} / {len(total_unique_ids)} ({overall_pct}%)")

    problems = [r[0] for r in rows if r[1] in ("FEHLT", "LEER", "KAPUTT")]
    if problems:
        print(f"\nProblematisch (fehlt/leer/kaputt): {problems}")


check_status()

## 2. Stats-Abdeckung: welche Statistik-Typen sind zuverlässig genug?

In [ ]:
def analyze_stats_coverage():
    stats_files = list(STATS_DIR.glob("*.json"))
    total_fixtures = len(stats_files)

    if total_fixtures == 0:
        print("Keine Match-Stats-Dateien gefunden.")
        return

    type_full_coverage = defaultdict(int)
    empty_files = 0

    for stats_file in stats_files:
        try:
            response = _read_json_robust(stats_file)
        except (json.JSONDecodeError, UnicodeDecodeError):
            continue

        if not response or len(response) < 2:
            empty_files += 1
            continue

        team_stats = []
        for team_entry in response:
            stat_dict = {}
            for stat in team_entry.get("statistics", []):
                stat_dict[stat.get("type")] = stat.get("value")
            team_stats.append(stat_dict)

        all_types = set()
        for ts in team_stats:
            all_types.update(ts.keys())

        for stat_type in all_types:
            values = [ts.get(stat_type) for ts in team_stats]
            if all(v is not None for v in values):
                type_full_coverage[stat_type] += 1

    print(f"Analysierte Match-Stats-Dateien: {total_fixtures}")
    if empty_files:
        print(f"Davon leer/ungültig (übersprungen): {empty_files}")
    valid_total = total_fixtures - empty_files
    print(f"Gültige Spiele mit Statistik-Daten: {valid_total}\n")

    print(f"{'Statistik-Typ':<28} {'Vorhanden (beide Teams)':>25} {'Abdeckung':>12}")
    print("-" * 68)

    sorted_types = sorted(type_full_coverage.items(), key=lambda x: x[1], reverse=True)
    for stat_type, count in sorted_types:
        pct = round(100 * count / valid_total, 1) if valid_total > 0 else 0
        print(f"{stat_type:<28} {count:>25} {pct:>11}%")


analyze_stats_coverage()

## 3. Feature-Sanity-Check: `match_features.csv`

In [ ]:
def quick_check_features():
    if not FEATURES_FILE.exists():
        print(f"{FEATURES_FILE} existiert noch nicht - erst build_features.py laufen lassen.")
        return

    with open(FEATURES_FILE, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    print(f"Total Zeilen: {len(rows)}")
    print(f"Spalten: {list(rows[0].keys())}\n")

    results = Counter(r["result"] for r in rows)
    print("Verteilung Zielvariable (result):")
    for k, v in results.items():
        print(f"  {k}: {v} ({100 * v / len(rows):.1f}%)")

    missing_possession = sum(1 for r in rows if not r["a_possession_avg"] or not r["b_possession_avg"])
    print(f"\nZeilen ohne Ballbesitz-Daten (a oder b): {missing_possession} ({100 * missing_possession / len(rows):.1f}%)")

    missing_elo = sum(1 for r in rows if not r["elo_diff"])
    print(f"Zeilen ohne Elo-Differenz: {missing_elo} ({100 * missing_elo / len(rows):.1f}%)")

    print("\nErste 3 Zeilen als Beispiel:")
    for r in rows[:3]:
        print(r)


quick_check_features()